In [ ]:
from datetime import datetime

from src.ingestion.pipeline import ingest_table
from config.ingestion_config import (
    BRANCHES,
    TABLE_CONFIG,
    CATALOG,
    LANDING_SCHEMA,
    BRONZE_SCHEMA,
    RAW_VOLUME
)

In [ ]:
#generate one batch ID for this ingestion run

batch_id = (
    "Batch_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
)

print(f"Starting ingestion")
print(f"Batch ID: {batch_id}")

In [ ]:
spark.sql(

    f"CREATE SCHEMA IF NOT EXISTS "
    f"{CATALOG}.{LANDING_SCHEMA}"
)

spark.sql(

    f"CREATE SCHEMA IF NOT EXISTS "
    f"{CATALOG}.{BRONZE_SCHEMA}"
)

In [ ]:
results = []

for branch in BRANCHES:

    print("=" * 70)
    print(f"Processing branch: {branch}")
    print("=" * 70)

    for source_system, tables in TABLE_CONFIG.items():

        print(f"\nSource system: {source_system}")

        for table_name in tables:

            print(
                f"  Ingesting: "
                f"{branch}/{source_system}/{table_name}"
            )

            try:

                result = ingest_table(
                    spark=spark,
                    catalog=CATALOG,
                    landing_schema=LANDING_SCHEMA,
                    bronze_schema=BRONZE_SCHEMA,
                    volume=RAW_VOLUME,
                    branch=branch,
                    source_system=source_system,
                    table_name=table_name,
                    batch_id=batch_id
                )

                results.append(result)

                print(
                    f"  SUCCESS: "
                    f"{result['record_count']} records"
                )

            except Exception as e:

                results.append({
                    "branch": branch,
                    "source_system": source_system,
                    "table": table_name,
                    "source_path": None,
                    "bronze_table": None,
                    "record_count": 0,
                    "status": "FAILED",
                    "error": str(e)
                })

                print(
                    f"  FAILED: "
                    f"{branch}/{source_system}/{table_name}"
                )

                print(str(e))